# Signal Hunters R2 - run 17 (teammate B): CNN v3 (LEVEL channel), normal canvas (FAST ~2.5 h), more ETI injection, stacker v9 pseudo-labels - EfficientNet-B2 + B0
GPU T4 x2, input: signal-hunters-r2, Internet ON. RUN WITH: Save Version -> Save & Run All (Commit).

In [ ]:
%%writefile /kaggle/working/cnn_r2.py
"""Signal Hunters R2 - CNN on the 'global-time canvas'.
Each cadence -> one 128x128 image: the 6 panels placed on ONE time axis including the slew gaps
(panel p row r at t = p*(16+gap)+r), so a true ETI drift is a single straight line through the 3 ON panels.
Input channels: z (bandpass-flattened, noise-normalised), zr (z minus per-row median: kills broadband rows),
on/off indicator (+1 ON rows, -1 OFF rows, 0 gaps), valid mask.
Domain shift GBT->PKS is attacked with augmentation: frequency zoom (PKS drifts look ~0.64x as steep),
SNR loss, dropped rows, broadband glitch rows, per-panel gain, frequency shift/flip.
Usage: python cnn_r2.py DATA_DIR EPOCHS SEEDS [arch=res|timm_resnet18|timm_effb0] [PL]
Writes oof_<tag>.npy (18000x9) and test_<tag>.npy (10000x9) to ./out/"""
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn, sys, os, time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
LABELS = ['eti_candidate', 'frb', 'pulsar', 'scintillating_star', 'satellite_pass', 'radar_chirp',
          'rfi_intermittent', 'rfi_persistent', 'noise']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
UP = int(os.environ.get('UP', 1)); LVL = int(os.environ.get('LVL', 1)); P_ETI = float(os.environ.get('P_ETI', 0)); P_TRAP = float(os.environ.get('P_TRAP', 0))

def normalise(X):
    x = X.astype(np.float32); m = (X > 0); x[~m] = np.nan
    x = x - np.nanmedian(x, axis=2, keepdims=True)
    s = 1.4826 * np.nanmedian(np.abs(x - np.nanmedian(x, axis=(2, 3), keepdims=True)), axis=(2, 3), keepdims=True) + 1e-3
    z = np.nan_to_num(np.clip(x / s, -10, 60))
    return z.astype(np.float16), m

def load(D, split):
    X = np.load(f'{D}/{split}_cadences.npy'); meta = pd.read_csv(f'{D}/{split}_meta.csv')
    return X, (X > 0), meta.slew_gap_rows.values.astype(np.int64), meta

def norm_t(x, m):
    """FIXED normalisation (v2). Old version subtracted each channel's median over time, which ERASED every
    non-drifting line (persistent RFI, non-drifting traps): a 10-sigma persistent line became 0.4 sigma.
    Now: bandpass = per-panel time-median spectrum smoothed ACROSS FREQUENCY (median, 9 channels), so narrow
    lines survive. Channel 64 (GBT-only centre spike, absent in PKS) is blanked in every cadence."""
    x = torch.where(m > 0, x, torch.full_like(x, float('nan')))
    x[..., 64] = float('nan')
    spec = torch.nanmedian(x, dim=2).values                                   # (B,6,128)
    spec = torch.where(torch.isnan(spec), torch.nanmedian(spec, dim=-1, keepdim=True).values, spec)
    pad = Fn.pad(spec, (4, 4), mode='replicate')
    band = pad.unfold(-1, 9, 1).median(-1).values                            # smooth bandpass
    r = x - band[:, :, None, :]
    f = r.flatten(2); med = torch.nanmedian(f, dim=2, keepdim=True).values
    s = 1.4826 * torch.nanmedian((f - med).abs(), dim=2, keepdim=True).values + 1e-3
    z = ((r - med[..., None]) / s[..., None]).clamp(-10, 60)
    # v3: LEVEL channel = what the per-panel bandpass removed, relative to the OFF-panel bandpass.
    # A scintillating star brightens the whole ON panel; the per-panel band absorbs it, so z alone cannot show it.
    bo = band[:, [1, 3, 5]].median(1, keepdim=True).values                    # (B,1,128) OFF bandpass
    so = s[:, [1, 3, 5]].median(1, keepdim=True).values                       # (B,1,1) OFF noise scale
    E = ((band - bo) / so).clamp(-10, 30)                                     # (B,6,128)
    return torch.nan_to_num(z, nan=0.0), torch.nan_to_num(E, nan=0.0)

def byte_aug(x, m, k_lo=5., k_hi=30., p=0.8):
    """PKS-like receiver noise in the POWER domain: power *= Gamma(k)/k  -> skewed log-noise like the test
    telescope (measured: noise MAD 15.6 vs 11.9 bytes, more negative skew). Also extra dropped/glitch rows."""
    B = x.shape[0]
    k = torch.empty(B, 1, 1, 1, device=x.device).uniform_(k_lo, k_hi)
    g = torch.distributions.Gamma(k.expand_as(x), k.expand_as(x)).sample().clamp_min(1e-4)
    sel = (torch.rand(B, 1, 1, 1, device=x.device) < p).float()
    x = x + sel * 24 * torch.log2(g)
    drop = (torch.rand(B, 6, 16, 1, device=x.device) < torch.empty(B, 1, 1, 1, device=x.device).uniform_(0, 0.03)).float()
    gl = (torch.rand(B, 6, 16, 1, device=x.device) < 0.01).float() * torch.empty(B, 6, 16, 1, device=x.device).uniform_(10, 50)
    x = (x + gl).clamp(1, 255)
    m = m * (1 - drop)
    return x, m

# ---------------- GPU-side augmentation + canvas -----------------
def augment(z, m, train, E):
    """z,m: (B,6,16,128) float on device"""
    B = z.shape[0]
    if not train: return z, m, E
    # SNR loss + extra noise (PKS noisier): z' = a*z + sqrt(1-a^2)*N
    a = torch.empty(B, 1, 1, 1, device=z.device).uniform_(0.75, 1.0)
    z = a * z + torch.sqrt(1 - a ** 2) * torch.randn_like(z)
    # per-panel gain on the signal part
    z = z * torch.empty(B, 6, 1, 1, device=z.device).uniform_(0.8, 1.2)
    # frequency zoom (compress/expand channel axis), p=0.7, s in [0.55,1.1]
    sel = torch.rand(B, device=z.device) < (0.15 if not LVL else 0.0)
    if sel.any():
        s = torch.ones(B, device=z.device); s[sel] = torch.empty(int(sel.sum()), device=z.device).uniform_(0.85, 1.15)
        xs = torch.linspace(-1, 1, 128, device=z.device)
        gx = xs[None, :] / s[:, None]                                   # sample positions in input
        grid = torch.stack([gx[:, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=z.device)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, mode='bilinear', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        mm = Fn.grid_sample(m.reshape(B, 1, 96, 128).float(), grid, mode='nearest', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        outside = (gx.abs() > 1)[:, None, None, :].expand(B, 6, 16, 128)
        # bilinear interpolation shrinks noise std -> renormalise per panel, fill outside with fresh noise
        zz = torch.where(outside, torch.randn_like(zz), zz)
        mm = torch.where(outside, torch.ones_like(mm), mm)
        sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        z = torch.where(sel[:, None, None, None], zz, z); m = torch.where(sel[:, None, None, None], mm, m.float())
    m = m.float()
    # frequency shift (roll) and flip
    sh = int(np.random.randint(-30, 31)); z = torch.roll(z, sh, -1); m = torch.roll(m, sh, -1); E = torch.roll(E, sh, -1)
    fl = torch.rand(B, device=z.device) < 0.5
    z = torch.where(fl[:, None, None, None], z.flip(-1), z); m = torch.where(fl[:, None, None, None], m.flip(-1), m)
    E = torch.where(fl[:, None, None], E.flip(-1), E)
    E = E + 0.15 * torch.randn(B, 6, 1, device=z.device)                      # per-panel gain jitter on the level
    return z, m, E

def up(C):
    return Fn.interpolate(C, scale_factor=UP, mode='nearest') if UP > 1 else C

def canvas(z, m, gap, E=None):
    """(B,6,16,128) -> (B,4+LVL,128,128) global-time canvas"""
    B = z.shape[0]; H = 128
    zr = z - z.median(-1, keepdim=True).values
    C = torch.zeros(B, 4 + LVL, H, 128, device=z.device)
    p = torch.arange(6, device=z.device); r = torch.arange(16, device=z.device)
    t = p[None, :, None] * (16 + gap[:, None, None]) + r[None, None, :]          # (B,6,16)
    t = t + ((H - (6 * 16 + 5 * gap)) // 2)[:, None, None]                       # centre vertically
    bi = torch.arange(B, device=z.device)[:, None, None].expand(B, 6, 16)
    sign = torch.tensor([1., -1, 1, -1, 1, -1], device=z.device)[None, :, None].expand(B, 6, 16)
    C[bi, 0, t] = z * m; C[bi, 1, t] = zr * m
    C[bi, 2, t] = sign[..., None].expand(B, 6, 16, 128); C[bi, 3, t] = m
    if LVL: C[bi, 4, t] = 3 * E[:, :, None, :].expand(B, 6, 16, 128)        # x3: same scale as z
    return C


def inject(z, m, gap, yb, p_eti, p_trap):
    """Physics-based synthetic signals (labels follow the priority order):
    - ETI: a narrow drifting line on ONE straight track in global time, ON panels only -> label eti_candidate
    - trap: ON-only lines that do NOT connect (random offsets) or do not drift -> rfi_intermittent
      (only injected into noise / rfi_persistent cadences, whose priority is lower)"""
    B = z.shape[0]; d_ = z.device
    t = (torch.arange(6, device=d_)[None, :, None] * (16 + gap[:, None, None]) + torch.arange(16, device=d_)[None, None, :]).float()
    tc = t[:, [0, 2, 4]].mean((1, 2))[:, None, None]
    ch = torch.arange(128, device=d_).float()[None, None, None, :]
    on = torch.tensor([1., 0, 1, 0, 1, 0], device=d_)[None, :, None, None]
    r = torch.rand(B, device=d_)
    e = (r < p_eti) & (yb != 0)
    tr_ = (r >= p_eti) & (r < p_eti + p_trap) & ((yb == 7) | (yb == 8))
    if e.any() or tr_.any():
        drift = torch.empty(B, device=d_).uniform_(0.15, 1.3) * (torch.randint(0, 2, (B,), device=d_) * 2 - 1)
        c0 = torch.empty(B, device=d_).uniform_(20, 108)
        amp = torch.empty(B, device=d_).uniform_(0.6, 3.0); wid = torch.empty(B, device=d_).uniform_(0.5, 1.2)
        centre = c0[:, None, None] + drift[:, None, None] * (t - tc)                      # (B,6,16) connected track
        kind = torch.randint(0, 2, (B,), device=d_)                                       # trap type
        off = torch.empty(B, 6, 1, device=d_).uniform_(-40, 40)                           # disconnect offsets
        tr_centre = torch.where(kind[:, None, None] == 0, c0[:, None, None] + off + drift[:, None, None] * (t - t[:, :, :1]),
                                c0[:, None, None] + 0 * t)                                # non-connecting / non-drifting
        cen = torch.where(e[:, None, None], centre, tr_centre)
        line = amp[:, None, None, None] * torch.exp(-0.5 * ((ch - cen[..., None]) / wid[:, None, None, None]) ** 2) * on
        sel = (e | tr_)[:, None, None, None].float()
        z = z + sel * line * m
        yb = torch.where(e, torch.zeros_like(yb), torch.where(tr_, torch.full_like(yb, 6), yb))
    return z, yb

# ---------------- model -----------------
class Block(nn.Module):
    def __init__(s, i, o, st):
        super().__init__()
        s.c1 = nn.Conv2d(i, o, 3, st, 1, bias=False); s.b1 = nn.BatchNorm2d(o)
        s.c2 = nn.Conv2d(o, o, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(o)
        s.sk = nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o)) if (i != o or st != 1) else nn.Identity()
    def forward(s, x): return torch.relu(s.b2(s.c2(torch.relu(s.b1(s.c1(x))))) + s.sk(x))

class ResNet(nn.Module):
    def __init__(s, w=40):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(4 + LVL, w, (5, 3), 1, (2, 1), bias=False), nn.BatchNorm2d(w), nn.ReLU())
        s.body = nn.Sequential(Block(w, 2 * w, 2), Block(2 * w, 2 * w, 1),
                               Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1), Block(4 * w, 6 * w, 2), Block(6 * w, 6 * w, 1),
                               Block(6 * w, 8 * w, 2))
        s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(16 * w, 9))
    def forward(s, x):
        h = s.body(s.stem(x)); return s.head(torch.cat([h.mean((2, 3)), h.amax((2, 3))], 1))

def make(arch):
    if arch == 'res': return ResNet()
    import timm
    name = {'timm_resnet18': 'resnet18', 'timm_effb0': 'efficientnet_b0', 'timm_effb2': 'efficientnet_b2', 'timm_convnext': 'convnext_nano', 'timm_effb3': 'efficientnet_b3'}[arch]
    try: return timm.create_model(name, pretrained=True, in_chans=4 + LVL, num_classes=9)
    except Exception as e: print('pretrained download failed, random init:', e); return timm.create_model(name, pretrained=False, in_chans=4 + LVL, num_classes=9)

def predict(net, Z, M, G, tta=True):
    EE = None
    if isinstance(Z, tuple): Z, EE = Z
    net.eval(); out = []
    with torch.no_grad(), torch.autocast(dev, enabled=dev == 'cuda'):
        for j in range(0, len(Z), 512):
            m = torch.from_numpy(M[j:j + 512]).to(dev).float()
            z = torch.from_numpy(Z[j:j + 512]).to(dev).float()
            if Z.dtype == np.uint8: z, E = norm_t(z, m)
            else: E = torch.from_numpy(EE[j:j + 512]).to(dev).float()
            g = torch.from_numpy(G[j:j + 512]).to(dev)
            p = torch.softmax(net(up(canvas(z, m, g, E))).float(), 1)
            if tta: p = 0.5 * (p + torch.softmax(net(up(canvas(z.flip(-1), m.flip(-1), g, E.flip(-1)))).float(), 1))
            out.append(p.cpu().numpy())
    return np.concatenate(out)

def pks_like(Z, M, seed=0):
    """deterministic 'PKS-like' version of held-out GBT data for shift-aware validation"""
    torch.manual_seed(seed); out_z, out_m, out_e = [], [], []
    for j in range(0, len(Z), 512):
        x = torch.from_numpy(Z[j:j + 512]).to(dev).float(); m = torch.from_numpy(M[j:j + 512]).to(dev).float(); B = len(x)
        x, m = byte_aug(x, m, 12., 12., 1.0); z, E = norm_t(x, m); out_e.append(E.cpu().numpy().astype(np.float16))
        xs = torch.linspace(-1, 1, 128, device=dev) / 1.0
        grid = torch.stack([xs[None, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=dev)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, align_corners=True).reshape(B, 6, 16, 128)
        out = (xs.abs() > 1)[None, None, None, :].expand_as(zz)
        zz = torch.where(out, torch.randn_like(zz), zz); sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        mm = torch.where(out, torch.ones_like(m), Fn.grid_sample(m.reshape(B, 1, 96, 128), grid, mode='nearest', align_corners=True).reshape(B, 6, 16, 128))
        out_z.append(zz.cpu().numpy().astype(np.float16)); out_m.append(mm.cpu().numpy() > 0.5)
    return (np.concatenate(out_z), np.concatenate(out_e)), np.concatenate(out_m)

def fit(Z, M, G, y, w_cls, epochs, seed, arch, evals):
    torch.manual_seed(seed); np.random.seed(seed)
    net = make(arch).to(dev); opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=0.02)
    bs = int(os.environ.get('BS', 128)); n = len(y); steps = epochs * (n // bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps, pct_start=0.15)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(w_cls, device=dev, dtype=torch.float32), label_smoothing=0.05)
    scaler = torch.amp.GradScaler(enabled=dev == 'cuda'); Y = torch.from_numpy(y)
    for ep in range(epochs):
        net.train(); perm = np.random.permutation(n); t0 = time.time(); L = 0
        for j in range(0, n - bs + 1, bs):
            idx = perm[j:j + bs]
            x = torch.from_numpy(Z[idx]).to(dev).float(); m = torch.from_numpy(M[idx]).to(dev).float()
            g = torch.from_numpy(G[idx]).to(dev); yb = Y[idx].to(dev)
            x, m = byte_aug(x, m); z, E = norm_t(x, m)
            if P_ETI + P_TRAP > 0: z, yb = inject(z, m, g, yb, P_ETI, P_TRAP)
            z, m, E = augment(z, m, True, E)
            with torch.autocast(dev, enabled=dev == 'cuda'):
                loss = lossf(net(up(canvas(z, m, g, E))), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); L += loss.item() * bs
        if ep % 5 == 4 or ep == epochs - 1: print(f'   ep{ep} loss {L / n:.3f} {time.time() - t0:.1f}s', flush=True)
    return [predict(net, *e) for e in evals]

if __name__ == '__main__':
    D, EP, SEEDS = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
    arch = sys.argv[4] if len(sys.argv) > 4 else 'res'; PL = 'PL' in sys.argv
    tag = os.environ.get('TAG', arch); SMOKE = int(os.environ.get('SMOKE', 0)); os.makedirs('out', exist_ok=True)
    Z, M, G, meta = load(D, 'train'); Zt, Mt, Gt, tmeta = load(D, 'test')
    y = meta.label.map({l: i for i, l in enumerate(LABELS)}).values
    if SMOKE: Z, M, G, y = Z[:SMOKE], M[:SMOKE], G[:SMOKE], y[:SMOKE]; Zt, Mt, Gt = Zt[:SMOKE], Mt[:SMOKE], Gt[:SMOKE]
    cnt = np.bincount(y, minlength=9); w_cls = (cnt.sum() / (9 * np.maximum(cnt, 1))) ** 0.5; w_cls = w_cls / w_cls.mean()
    print('device', dev, 'arch', arch, 'train', Z.shape, 'class weights', w_cls.round(2), flush=True)
    if PL:
        pl = np.load('out/pl.npz'); sel, ypl = pl['sel'], pl['y']
        print('pseudo-labelled test cadences', sel.sum(), flush=True)
    oof = np.zeros((len(y), 8 + 1)); oof_pks = np.zeros_like(oof); pt = np.zeros((len(Zt), 9))
    for s in range(SEEDS):
        for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(Z, y)):
            if os.environ.get('FOLDS') and str(k) not in os.environ['FOLDS'].split(','): continue
            Za, Ma, Ga, ya = Z[a], M[a], G[a], y[a]
            if PL: Za, Ma, Ga, ya = np.concatenate([Za, Zt[sel]]), np.concatenate([Ma, Mt[sel]]), np.r_[Ga, Gt[sel]], np.r_[ya, ypl]
            Zp, Mp = pks_like(Z[b], M[b], seed=k)
            pb, pp, pte = fit(Za, Ma, Ga, ya, w_cls, EP, 100 * s + k + int(os.environ.get('SEED_OFF', 0)), arch, [(Z[b], M[b], G[b]), (Zp, Mp, G[b]), (Zt, Mt, Gt)])
            oof[b] += pb / SEEDS; oof_pks[b] += pp / SEEDS; pt += pte / (5 * SEEDS)
            print(f'seed {s} fold {k}: F1 {f1_score(y[b], pb.argmax(1), average="macro"):.4f} | PKS-like F1 {f1_score(y[b], pp.argmax(1), average="macro"):.4f}', flush=True)
            np.save(f'out/oof_{tag}_partial.npy', oof); np.save(f'out/test_{tag}_partial.npy', pt); np.save(f'out/folds_done_{tag}.npy', np.array([s, k]))
        f = SEEDS / (s + 1)
        print(f'== after seed {s}: OOF F1 {f1_score(y, (oof * f).argmax(1), average="macro"):.4f} | PKS-like {f1_score(y, (oof_pks * f).argmax(1), average="macro"):.4f}', flush=True)
        print('per class (PKS-like):', dict(zip(LABELS, f1_score(y, (oof_pks * f).argmax(1), average=None).round(3))), flush=True)
        np.save(f'out/oof_{tag}.npy', oof * f); np.save(f'out/oofpks_{tag}.npy', oof_pks * f); np.save(f'out/test_{tag}.npy', pt * f)
    print('DONE', flush=True)


In [ ]:
import base64, os
os.makedirs('/kaggle/working/out', exist_ok=True)
open('/kaggle/working/out/pl.npz','wb').write(base64.b64decode('UEsDBC0AAAAIAAAAIQBonKL5//////////8HABQAc2VsLm5weQEAEACQJwAAAAAAAJQFAAAAAAAAnVm7bmRFED2d8hXODJIDNuUDyEAkBETIsEYEiEX2igT4Cn4Yr3e6+7x6vOLKO3Nvd3XVqapTdbtn//32+2+++2HgT/x1+/bh6efH269ubv/+6c3t3c3tL+8e3z/e//7ju8e3Dx/Gv77/7enhefzp1/s/Hp6fP3/z5fN198XdzT83//P6bBwvXJl4mQM+/M1BxArwAPjDhEGzSxiuQXFBR+YKnU3RLYTpA0uYKNwNjKKbBKAWSbx6w6hfgglTHwDFAx5S1TSlvh2TKvlgfR7Dgz+HOFZNh7S4NrphpvFtcQEqPA1Om1gRrVkdy/stAOZLsQoslI1+SRr2AasgTEdnP4PYOrBkF6kxhud+sPj82ow9paDjZnVTIfuwaxqGo6rg/gBHm647R5Fw0R9Ji8QyME0TO1qr2C+B41hvei2nBU8LLrj8l1ilGdpjya51Aeaw2V0oY7GFX1wY4RYvyh6wGXqJnQFUtygAvBz7H4QdLImVlhIOdsJqlXk3JUCC3gYEoebVmCrqNWwtv471MG13B9p7rSjiMLGoHLprvmbA1RSTeLDnzvslBe7VBn4vBoSTUUvmY/pFDxHkXdLp5oVuFEuLnraAZpdjVDJb3zHiHrN1Ws86pUzbHQNpebiIUud5iTZ2irMdsGp/01LGI6/sskUEENq03aQuYWJxpdUd45hFPSbfDjHeLvSOTVKXAPnLrWgks16dvvFl5kt9RTR6DQg6Ejywc5CjVtO4kL9Y8DJyfhxKE2feiJZAT57pkK1ywaIe5MNHYIDlOgLbDTdbEDNq2LKYcNE/F9Nec6q7fcbd1x6aazRprJXMmDTqdPAjFWxYbVaCR9vrZUANhdo0aU5odWc6V6iGzPGK8kHGhFcOQKaxc4MhTqyFJU6JgYEztJMsQZCwyJu3+DVleb9Eq5i4dLZtkRJgEG1utDx9NHSoAK1uhu5EikVcKvEajHhQvlDljgAJXw5rwTRdzCQ/d5ATrPsToLx2SRch5s4xzJc6Z5Iq1ov96NpiQ+sxNHnNB2VTbIKREmXyBcwrPBNofIsqeC3gB0OH8rEdBvO9Q66tvgzlBqciWHTNbEQLgamS7GoUKWG+lgbamc80xXnuTPPW8mRk7pHpKYVInTBBd4x+huT1pe9caycUVc+YgFdsAmy0S+GmjUzf3jlKzbt9hHc7UL1cts7GFYrYObnBcYvVVCHxEE7nzEiDTF0+4+5sT/Jcow7HsbOkcK+c/ERDviOQ4Ry7QZQUbcLMZEjaz/Wz7OWWrErWIWka1c/oGnTAdHPQIXGgEEWneHE7CkK2mgjySH1sfzSOlNMTr0vzDPwJWhUQx+yXBo14xvwai6cPk3yttK8nOjuAlKKerfiUaRSrFzVAC3nAqBD17tW24IYNqte4c50elTXLVQV+NA/ORztnze9DapIKTazPIQT4nFFMdk0p5s0kT5WS5wy2PQc5J38bU9SxCaPwm5YHg2oVUelThwX7UGPh45/ASVo9IXoLENxH4oyxO24eSRC3XOqcGuh3tI5d65e3rO5Jt4whpXebg2PvMA6N7UB9qAr5yasr8sUXm4U3LWo+Av1bflayLX8pzyvYwVHJeZ66RCFhmRh4Yr772YY4HNXgiV2wzSUlZinNQ8nNiMWmVvQCIj78SQzIBmetB3F0J/paGbUxRCGepUEldb6y6ih3KnUAtegGj8h1iBHLsRIh1KB1tSzKHeSnpLnDiTrsZtDgjdljYJJ7vRzodI8sMS6kljIFQSY2GWOssA1W1K486onPqskYffXUvEFEBrY65LQoZjBZDxTENXJZBNVb9jzo2DkMFKlXipGZWbEuT0oFHQpj/UebZpu0g2ZDS3SC+CmK6CioUT703tT0bMVvFAfibbSrZPRcdpJPo9MMppK+mAu4zkPnMf4DUEsDBC0AAAAIAAAAIQALqzAK//////////8FABQAeS5ucHkBABAAcP8AAAAAAAAqFgAAAAAAAK3cTcstVxHF8c7LjcFPkdlVyMSBEMSxM8WJA0cS9IqCGEnEifgp/MJGzJ4s+PHf58aePHSfPt37pWrVqlV1nn//6je//PVvP3j+8fzz7R/effP7r9/+7LO3P//zF28//+ztH7/6+u9ff/nX33319R/e/ff6L778yzfvvr3+zZ++/Nu7b89/9MVPfvrF5z/+/LN/ffa+xw8/fv53fPLd3w+/+/vRd39/MH8/nfs/mc/fzPVz/4dx/dP5u8/d8e39b+b+/d6Z54dzvvd9Op9/MH93Hnr+jm/XeZ97xqF13++9mfN9785vP9f5rvvOY/dn3/cR7tvnnWPnIbvSOuy4dh67vtoXfU/PkR3s/eVf+73z+Y53n6P9OM8tv9pz2ee5fvZJ+7p+9WbOd57Cif287EN4sut8Pj/j1fx23fSej+O+Ha9wSu9dPxf+LW4sHgjfZBc7vt3Xxac933V4cH39Sfu6diacXfuUvWp/1x6Ea9pX2bnsfvd97XD3Ye1h7bzis+Lp4qj8fcel9+znwue1c/GIsl/tw+JGPW/t6hzCU8WB8nutx/m7+6Hr9by1h/Uf2e35ftmp5r+fn/dsvNv9V/ze8Wod1k+K3yjub9za9yseLV9cnrNxSLik9bld1/3+4pZ40+Jc8UDFccVp4cHa3/rvLW4pvu/9a99r1+W/wmeti+LGLe8rPiOetOuhvEB+JTvcfVL8X/vTuexEcX/Xc/dfvEJ2onONZ98jfC48lh08cy4etnmR7FdxuXjdLV7u/YobFa81r+UD5YdaZ+kTOt+4v/sjPiC/3riydrLP3feWn+38bvF653vLF8XTlH+IZ2ichZOF19/XLrWva6/LL3ee2q9nzm959DmEg8LTM45nzpUfrz8qP5K/Cicrb979Et+9zcPE7wrXhZfSARW39/yZ67JL8crd74rbik9aP33vHNov4W3xR+Hm2pv0Ns1HfHLXvfKExTfpRcoTF7/Xvne++9zFMa3D+vHev8+T/nWbH8sfxfNr3+v58t9X8/DSRdZexeOlP8k+xbvFbxUP1k7OsftcOLc4/NH81TpKv9G6iDfsOPY59bziT9LjFMe0j+JVa78VJwuvb3UR4WXxrMJr8R7lHcqbFGekO+36So+Unew4xfMXfzWPyvu0jpWXVX6p+C47lv2tnZ55q/5ZOCT8r7ih8ZQ+p7pm4W/lF2uH0pnEAxYfSmfZcVS8ULxWvUj5h/T0Z+7bdRE/kZ4m+1Ze96ruovXQvJXfFt8SHmvf5C+VZ5b/io8KBxX/hcdr/+LXiwuF/+Lf0kcUXxY3K99U/indpPin8mDhaOG09nH9V7yi4ufay8bDisviZ7e8uepRpR+Jh1aeqvxOeL73aV7K34pHa17iVforO1ecqzpO6aH6nvJm8RDFydIDld+sTnKOqhtIr9U+lp6z63Oer7rP4ujuz46v6mTy533P8qJdd+GOePKuz+rOipPi8bu/4vPnEG4Wr9A8hWulP2vfxMPUP1D2KH1r93Xz81of1S3eFydL5604Knx5Lu+rfs39vnBQ/q766p6rjqz3av/XTkon1/5W3iv9QHnIxrNzFA5Ip3rm8+IZWre6Xv0A4tu37y+/LF1F/i9dt3Bmnysef5sX13gqj1O8LJ4jPiz71vgfXFe8Wj4gvW7neY7bPGfX75n7So8vvbnqRaqTy16V32s+slPx9bKnxU/hTtXbhG/iXWtnqrdLx37mfN/zzLnWuerrO79nPldeKx4pnFD9efPg1QtlZ+IH0q8Wz4qv3fbLvppHq54vO9e8bvWn2m/x/qqPl/1WPVs4uvWiqnMqr1j8qr7G4oHnUN4p+66+EsWj4r/Ct9Jrah/Ev+o5ilNap2c+V7wsvlp4dI7K69ae9rpwdHFG9rf2u4fihtZZed7yyOL3sjfp9BXndv5VPy5dS+spvq3rtR/iH2X/qm9IJ5I+WX1k0se0jq/ypBr/2sWuj+p8iwPVj6D5rp0Jh8ofNW/pA1VHue3/kt6gOs++57YvS3UG5WGK97Jb6UyF3+I9O57qJ31VhyudVrpr+ZF4vPZB9iAcV3za/LR0qrXfOheOCCeF+7s/smvxKfmf8Ep4rj6n4n+VL4q/Sq+XHSjPLh606yF72vftOlXdoOoalVfsPkkvUP6kPFJ6x9qHeJPq2IVr6lPR79HUN7T2s+tbvKJwVPm/1lv5pZ77zPXid6/mKYozwrXyK+X9pZNqP9duqr+pdFXpGMIl8Rbht8YhPBOvqjixeKG+w8oD9rzGe1vPU/wSH6v6leYl/lP8UTpI6Y6yN42r9Ebh3Y6ndKnSsW51nOKrO17pcuo70zhlh9UXVvlI6Sa7L5rvOdZexPOFd5U/ln2WHyi/Ea9XP5NwRvF010n6aPHW/bv1843Two3ijdqPvW/jqfBF13efnjlXnFI+pvWs+t+ud8WLHZ/yDuFF5c3imYrHt7rYPld+VPWT4ovC+eL14ue7DjXOihfSKWUX8pcH11X/VD/X4k7Fce2zePraaekKtU/iYYX7i1OVd0vPuK2blU6l/En+Kh5c4928S/FYeU/FaY236sLCuVv9avFDecQ+T/Yo3nOOwj3h9y2vKv9RvDiH8kV971W9WXmW8K10avnL4pbwcHGw9DzpBYuLijs7nrIf5Vm38V3PVx+b8jbZleKi7PE2fgrn1E9b+cyO4xy1X5U/l364+6Q4LbyTTlE6TNUhdlyFOxV3pXOUXbyKb7Jr6fHioWWHix87z6pPax2kp8p+9/nK85RPlQ5eeC/7FF/ddVod4rYfQHFfuufi8M5f+du+X3rRbZ5VfFT51vIq7Yt0v/2e6nrif/t3/Ul55+LaM/eJJ4k/S38qfrDjUz9w1VGq/rjxQ+M9h/iNeIDyM+n28uPbvPoc+33hvfBBuLDxVXyq4rbi5T6/cEzj3vUQXixeK24Jx+XX67+q5yi+1bpVPiGeJN1KfqznbXyq+Cp7X/sUHu73b/UmxSHhwav5cMWJW96j/Vfc1D4p/qi+K14oPqU6Qv0eqfhF5R/6vwS17jt+6UnFR3fdld8oz9P3d11Ll1U8lF2LrypeS7cuXUk8Wfu8eHkO9b1rv8Qnym7Vt1V6lvIy6Yz1e1rxlZ1n6ebnqH7uiruav/x+ebryofVj8Zpap+qTVJ1XeYjwXTqn8EY4u3Fyny8+L14jHUe4sfxV6yt/kj5VfiDepH0of9bv/27rx7sv4puqw9Z+V16o8crO1d9TelbxEfn1jk/+XOtX81I+ftt3onlq38SflD/JvqTLabzlz9WXtH5Uuo90r1sdXfxbuKp65m1etn/X7oS/yqeEV6r7Ka4L11QnrbqJ8EE6ednD2rf0n8qjbvmS8O6W3y8/0HqXn4kfKX6UTq98p+KR8lz9DkRxXv0N2s+d7zm0nsLnyvOFS7KLxfXdb+3z7nfpLxqX+rt2Pc9x+zsb4bzqEsKr2h/lzxt/K+8Tz951qXEUfux9im+KQ5UXimcob9R89r3nUHxev9t9k46m+e9zS+9YO9I893kVf4W7VWcU7ktXL7soXfC23qb6ReV5t/XOZ+6TPyveV15S//9MOLvjvPUjreMz16sOWDpa4YXyaOFf2bfsofQ44f/yBtWbipernrX+VHxe+FZ9Nqpn3OYlu863dRzxPuWb0ovVf3Mbn1T3FM8qnizcrXgo3KrnSL+oOrbO9T7V77Rf4pHqJypdQnWytY/1t923wuHSQ0rX0/roveIJ1c8r3eUc8pvFkXNUX0HhofJp8THNu3Qe+W/xcOXPhcO13+qfkD2pTr3z1njkF9UPKrze+C77VJ5WvO5Wb9X8ZPeLK5qX7Ll4X+mI4uvKS/7fuCP9+Znz1aP2uvJ84XPVw7XeNR/lr7f8bfdx8a/0hV0/6SqKw9KNqw6oPF56ZOGh7Eu6s/BN+Ct70bl4a/Wj1zgXB0pvEd5p31U/W7vaenTpsuLvypeEk9LZ5Qeyk9t9XL9Tn5vw8la/qX2rc71HdQDx1MpPq69EeZv4UP2/6PJr6e2Km9LbF4+rD+Y2/9E+ibfoffq95Dmqrlz8tM6rv2r3e9f7tj4uv1N8OYfijPIUvV88Sbh1q3epj6j6XqqPTrhVPFP+KX618xN/Fy4rDxHuaT2LB2x9ae1oxyl8r3xfedfiSvXL6DmKQ/pdgfKTZ86lG6wdlR8p7639WfzTPu1+6r2Kt+LVz5yrXiqcUN5ecUHrqzpJ5RfS/xQnSwfUukpvUnwTD9I+Fc4qDmpfxCf1V/FJ+YHyiZrnxuvqL7idR9XjCmeEZ6qzl79VvWb1Ou1D6Z1Vz1PfatmN+ILWXzi589h11z4VnhavVT9c9TO92s9Y9S/5qfBJz5cdSocRPur/A5TuIl1g9/196xTibcrzKx+pOFR5s+5/9f/1FH4Ih1U3uc1HFNdVXyw/rbr+2tH+1e9kxGeVl2ve0k1URyu/VT+bcGL3f+PAq336ZS/iaRUfxTPq/14p79zxneO2bqm8SbxKdX/pMtLNb/v7lM8tTgmXNQ/tr/oEpBMtPkr/0LgLJ1QHqP+DUzxi7UD/f168RfhRvO8cwgvFN/VRFJ7v+8pu1PcsvKv8THFFOofmc5sfab+UzxfuaV3kZ+Lvpe88c/0c+5y1c81XeFfrXPXGxdldN/2+Vv4uPU28T3mN+gQqzopvVH+f6s6Fz8/cJ1yXDr9xo/LC4jnK6/V7FPFR1WOUV2i/93mvrv9tHFbcrfyr8nfZ0frNjvM2L9K+q29Wv7sonXjtb/dD8UT5bOVV8qMd7/qxeGj1RVS+VHFI/Ft5jHSHyuOqnqT8TX0vit87DtXzhOfCW+kIiptVd1xetPYt/lX9Sa/2k7y6LxWXKx7vOgu/lTfJ7sSLte+yI+WrquOo7iO/X39SXBfeVJzUekhP3fvOsXi+6yOcXP60/LDiXuHBjkN4KnuoOK46kvjnrU4n+1I/mXjOOXac51Aetnhafr7+pfxV+PmqfZQOU3mO+ozWjm91QNmH7OIcykuk6wunql60z5Uf6Vy86pnz1R0XJ+p3faXPKN5VXC9dq/jgM/ft/tzmz4tn0n0qb1VfzTmE57XPlU+JV93G113njV/KCypf1Pz0+yDFv/VPxQfFe/Er6V7S31U3qHWseFD68sZdjV91CNlH6S3rj+I/wj/5pfZJ+dc+v3Thivfyj9Lnat12f1Tv1Liqj1JxT/xu10f6qnTWspviYcrbNI5zlD0pvkgXVl1vz+V/2o/171frsatv3fL8suPitc9cl79oP5dvFP/VeIWXt/8XUTisfO8cpScVD1q7F44I/8UH9/zBdd2vvj/Va+RXpS8UT9n91HtKD1JerLxa/qR9Ek6co37fp/V45vquwzmkBxd+Sl/UuqsPp+YjfN+4onqA9k288Byq37zKz2U/6gcuvyr7qfr24ov6QFR3Ld6ufEC4pf1/5j7lK8Uvdv63/iS9SDhV86u8Yj8/x+KC/LHwXTyi+kwUV3a9S+cW/361j0N17n1+9XmpHlt9iBtPtX/iQ7Xva+fCY9m3dN+yv33Ozu+2f1e6yjlUdxMPqrrnrZ6g+Vb+KrtT3q77NQ75t+Kbxq9+OenFhWNrx+JhwsXie2svle9JDxJ/Ul5ecWftUv+3o/o8yz53fcV3VJdW3il9snQT9a9rv9R/ovgv/lt1SMWFZ66r3lLxb+1v1098X3n0zvuZz/d9hevy/6ozyk6lP4hnVd6k/Eu6WuVd+9xn7qtxaF+rz0D9Fzv+soPlLeqvK14sOy5eJf2t9APhmHQx6SKaZ+nWyjdKzyo7VX9Y6U1rx5XXKZ4Lr4UX2v/SH8QDFTfED5XH7/fLznffxV+U17w677VXff997UD17uK9Nc7VI+Q38kPNZ++rOCy+U/WJzZ92/ZU3VTwuPiV/e+b6bf5T/qZ1Uz5QefPiifqYZPdaR+lTt3l88QvxBOkzspOKe7d6tOxD/Ru6X8+teHCruxUPXVwVP1t8f+Zz4X71oyiOFS8Qrj5zvfIwrWfVmfc556h8fHFyv6/6leKs4mXtf/Ur7XPU1yycq/gsP5KeJr9WnC5crLxAOCb/UJwQLqluID1D/fh67y3v2PUQj7sdv+xjcUZ1U/nhOYrvFk/b5+74n/lceo/49TOfa52km6qvsH5vUjqveLb2S/FGeHHG98x1xRnh8O37Co9VbxEvWPuQTn+Lb8pfdh1kr9VPqLgsHrb+Uuus/hjxOdnn7vfWqYWn0vGFq1rP4hc7L8VH8VH1l25+KT4l/yy+It5Zeo90RvWlSA8R7678RHnNrvPOR3yonqO8ofi18u/K4zQP5Ztr39p/1UGkb8gPhY/iebXO+17F1covpU9JP6x13fsX96V71/O0PsX3hf+Vb0tn1/uq/3jxU/hadlZ6TfFx5V3i6RqHcFn5iPR04YXyiLUH6VB6v/K86i9SPFB80P5X3271MUqH2ecXf9X6v9p3dMv/iw8J/4Xny/ur7qQ4pPqE+Oj6leoc4m3yL8Wl4nVa79LFpWuLx+36SffQeG79Wnqy1kl95Iq3wtnbukDhXOGf8m6Nb/dF+cjqg5VvPnNd8W7jfvXlrr+pDib9SetWus+O55lz1fWkEytPE+8Tjoo/KU9e3Fbck52qj0u6ffGI3Vf5tf4fp3BIOHYO8baab+kt4pfCD8X72893fNoH/T7ttp5/jh2//Fg8qvBs91U6XeGU9CDxj92v274m8e71f/F52UPxL+Hebd2x8mLZW/FrzU+6xPv2kUgnVRzc8Wnct/m98L7GW/qu4lbZVeW5VRdXvKn/h/F9+/PFk0oXWnwpve8cOy/FGcU/xW2Nc+1cfWG1X+rnKJ3iXH/m84ozt/6y9iKc17iUF8tfFM93nLvP8mf5teyk7PI2T1IdQTheebZwdOPoPl+8f7+/dll1qbIv6fG7XorHsuNb+1P8qfrRrpd0Qe3Pvld5pPQw5aPnqPyz7F1xUPspXb948/ZXPfN51cPlT7qv7KXycPUn6L3KI+r3hFrf9UfZXeVVpRdVf+W+r+q54t/CR+mpFUdqn26/X3pvxeXSzzYeiYdUP9bt96u+V7qS+IDyFeUVz5yrflp8T3mv/GfvO0fZr+oxynP3ffq++lpkP+XH6t+RDlp4rHUsPC3eJb8Sz7jNiyofFN6r7032V3qb3le8WefVJ6D9EX5XHqj8XH4h/r18QnikvFL1OvHX1bducUC6r+ypdAXlBaq3at+K35Sd7vgXJxQ/xG+1r+LjyiOU19/Gc/ED4abyR+1TjUc4vusmffqcbz9d1QO0P8onSqfRufD0FjfkTzs+jVs6/9rPrv/iQ9WXtc7Ff5ePSf/Z8a4/P3O98oXdT+nM+33VZfY5iuda352v9Gj58z5ncapwtvpE1McjHKz/8yR/uNWHhIvitfIf9aOUnrefP3MufV5+utelZygOVr/R+qfqftLHhZu39qo4Iv/TPmheyqv0/60qnxQ/UZ6luLX8bud1q6sIXwunpUNWX1zFc+Xt4sWK/7te60/7PO1f1fPEv9Y/tQ6rq+18xVvVByMed7uu9fuB0plezcsKf8SjCoeV9619yO8Vn+V3shfxMeGS6h07j6q76Dmy2/X3/wBQSwECLQMtAAAACAAAACEAaJyi+ZQFAACQJwAABwAAAAAAAAAAAAAAgAEAAAAAc2VsLm5weVBLAQItAy0AAAAIAAAAIQALqzAKKhYAAHD/AAAFAAAAAAAAAAAAAACAAc0FAAB5Lm5weVBLBQYAAAAAAgACAGgAAAAuHAAAAAA='))
import numpy as np; z=np.load('/kaggle/working/out/pl.npz'); print('pseudo-labels', z['sel'].sum())

In [ ]:
import glob, os, subprocess, time
os.chdir('/kaggle/working')
D = os.path.dirname(glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)[0]); print('DATA', D)
!pip -q install timm 2>/dev/null
EPOCHS = 30
p0 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=0 UP=1 BS=128 LVL=1 SEED_OFF=24000 TAG=effb2_lvlfast P_ETI=0.08 P_TRAP=0.04 python -u cnn_r2.py {D} 30 1 timm_effb2 PL > out/log_effb2_lvlfast.txt 2>&1', shell=True)
p1 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=1 UP=1 BS=128 LVL=1 SEED_OFF=24500 TAG=effb0_lvlfast P_ETI=0.08 P_TRAP=0.04 python -u cnn_r2.py {D} 30 1 timm_effb0 PL > out/log_effb0_lvlfast.txt 2>&1', shell=True)
print('started')

In [ ]:
import time
while p0.poll() is None or p1.poll() is None:
    for f in ['out/log_effb2_lvlfast.txt', 'out/log_effb0_lvlfast.txt']:
        try:
            lines = [l for l in open(f).read().splitlines() if 'fold' in l or '==' in l or 'Error' in l or 'per class' in l]
            print(f, '|', *lines[-3:], sep='\n  ')
        except FileNotFoundError:
            print(f, 'not created yet')
    print('-' * 60, time.strftime('%H:%M:%S'), flush=True); time.sleep(300)
print('BOTH DONE')

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile('r2_outputs_run17_B.zip', 'w') as z:
    for f in glob.glob('out/*.npy') + glob.glob('out/*.txt'): z.write(f)
FileLink('r2_outputs_run17_B.zip')